# C10-competition-craft — Practice p18 — Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804

df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y
)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=9)),
]).fit(X_tr, y_tr)

claimed = float(model.score(X_val, y_val))
honest_val_f1 = float(f1_score(y_val, model.predict(X_val), average="macro"))

model.fit(X, y)


def predict_labels(X_test):
    preds = model.predict(X_test)
    pretty = np.char.capitalize(preds.astype(str))
    return pd.Series(pretty, index=X_test.index)


probe = X.iloc[50:90]
broken_out = predict_labels(probe)
broken_checks = {
    "series": isinstance(broken_out, pd.Series),
    "length": len(broken_out) == len(probe),
    "index": broken_out.index.equals(probe.index),
    "vocab": set(broken_out.unique()) <= set(np.unique(y)),
}
violated_check = next(name for name in ["series", "length", "index", "vocab"]
                      if not broken_checks[name])

rubric_scores = {"W-A1": 1, "W-A2": 1, "W-B1": 0,
                 "W-B2": 1, "W-C1": 0, "W-C2": 0}
rubric_total = int(sum(rubric_scores.values()))


def predict_labels_fixed(X_test):
    return pd.Series(model.predict(X_test), index=X_test.index)


fixed_out = predict_labels_fixed(probe)
fixed_ok = bool(
    isinstance(fixed_out, pd.Series)
    and len(fixed_out) == len(probe)
    and fixed_out.index.equals(probe.index)
    and set(fixed_out.unique()) <= set(np.unique(y))
)
(violated_check, claimed, honest_val_f1, rubric_scores, rubric_total, fixed_ok)

### (b) Metric audit

For a classifier, `.score(X_val, y_val)` computes accuracy, so the submission mislabeled 0.806667 accuracy as macro-F1; the actual validation macro-F1 is 0.789356.

### (c) Rubric audit

- **W-A1 = 1:** the scaler, 9-NN hyperparameter, and all-12 feature set are complete.
- **W-A2 = 1:** judged on its own terms per the rubric — a validation score value IS present, so the criterion is met. The value is MISLABELED (accuracy passed off as macro-F1) — that failure is called out in the written audit (b), not double-counted here; the omitted carve/seed details belong to no W-A2 clause.
- **W-B1 = 0:** “strong algorithm” and “good number” state no property of this dataset.
- **W-B2 = 1:** the 2:1 imbalance and majority-only failure correctly motivate macro-F1.
- **W-C1 = 0:** no concrete alternative with an outcome or precise rejection reason is given.
- **W-C2 = 0:** “got it right” supplies neither a limitation nor a next step.

### (d) Corrected 6/6 writeup

**Approach.** I used a `StandardScaler` + `KNeighborsClassifier(n_neighbors=9)` pipeline on all 12 features. On a stratified 150-row validation carve with `random_state=20260804`, the recipe scored macro-F1 0.789356; I then refit the same recipe on all 600 labeled rows.

**Intuition.** The feature scales vary widely, so standardization keeps large-scale readings from dominating kNN's Euclidean distances and makes inspection-profile neighborhoods meaningful. Macro-F1 fits the roughly 2:1 imbalance because it gives struggling colonies equal class-level weight.

**Alternatives.** I rejected unscaled kNN because the large-scale noise columns would dominate raw distance, a precise mechanism-based failure. Limitation: (k=9) was not compared with a predeclared capped sweep; that is the next experiment, and any candidates should be logged because repeated validation selection makes the winner optimistic.

### Answer check

In [ ]:
assert violated_check == "vocab"
assert broken_checks == {"series": True, "length": True, "index": True, "vocab": False}
assert claimed == 0.8066666666666666
assert np.isclose(honest_val_f1, 0.7893564476296547, atol=1e-12, rtol=0)
assert rubric_scores == {"W-A1": 1, "W-A2": 1, "W-B1": 0,
                         "W-B2": 1, "W-C1": 0, "W-C2": 0}
assert rubric_total == 3
assert fixed_ok is True
assert set(fixed_out.unique()) <= {"struggles", "thrives"}